# 06 — Evaluate All Models on Clear-Day Test Set

Loads the best checkpoint for each model and evaluates on `clear_day/val`.
Also profiles hardware metrics (VRAM, FLOPs, inference speed).
Results are saved to `results/clear_day/`.

**Prerequisites:** notebooks 02–05 must have completed training.

In [1]:
from pathlib import Path
import json, torch
from dotenv import load_dotenv
from src.eval_utils import compute_map, compute_precision_recall, compute_per_class_ap, evaluate_rfdetr
from src.hardware_utils import measure_vram, count_flops_and_params, measure_inference_speed, measure_rfdetr_hardware
from src.data_utils import CLASS_NAMES
from src.train_utils import setup_logging

DRIVE_ROOT    = Path('/content/drive/MyDrive/FON/master_rad')
CONFIGS_DIR   = Path('/content/data_prepared/configs')
COCO_DATA_DIR = Path('/content/data_prepared/coco')
CHECKPOINTS   = DRIVE_ROOT / 'checkpoints'
RESULTS_DIR   = DRIVE_ROOT / 'results' / 'clear_day'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
from ultralytics import YOLO, RTDETR
from rfdetr import RFDETRMedium

yolov11 = YOLO(str(CHECKPOINTS / 'yolov11/run/weights/best.pt'))
yolov12 = YOLO(str(CHECKPOINTS / 'yolov12/run/weights/best.pt'))
rtdetr  = RTDETR(str(CHECKPOINTS / 'rtdetr/run/weights/best.pt'))
rfdetr  = RFDETRMedium(pretrain_weights=str(CHECKPOINTS / 'rfdetr/checkpoint_best_total.pth'))
rfdetr.optimize_for_inference()

[2026-05-10 18:55:12] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-10 18:55:12] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-10 18:55:13] [WARNING] rf-detr - Checkpoint has 10 classes but model is configured for 90. Using checkpoint class count (10). Pass num_classes=10 to suppress this warning.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [3]:
scores = {}
per_class_ap = {}

for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
    logger.info(f'Evaluating {name} on clear-day val...')
    results = model.val(
        data=str(CONFIGS_DIR / 'dataset.yaml'),
        split='val',
        device=DEVICE,
    )
    scores[name] = {
        **compute_map(results),
        **compute_precision_recall(results),
    }
    per_class_ap[name] = compute_per_class_ap(results, CLASS_NAMES)
    logger.info(f'{name}: {scores[name]}')

logger.info('Evaluating rfdetr on clear-day val...')
rfdetr_result = evaluate_rfdetr(rfdetr, COCO_DATA_DIR / 'clear_day', split='valid')
scores['rfdetr'] = compute_map(rfdetr_result)
per_class_ap['rfdetr'] = rfdetr_result.get('per_class_ap', {})
logger.info(f"rfdetr: {scores['rfdetr']}")

(RESULTS_DIR / 'scores.json').write_text(json.dumps(scores, indent=2))
(RESULTS_DIR / 'per_class_ap.json').write_text(json.dumps(per_class_ap, indent=2))
print('Saved scores and per-class AP to', RESULTS_DIR)

18:55:18 | INFO     | Evaluating yolov11 on clear-day val...


Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2392.9±836.8 MB/s, size: 69.2 KB)
val: Scanning /content/data_prepared/yolo/clear_day/val/labels.cache... 1764 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1764/1764 284.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 8.8it/s 12.6s<0.1s
                   all       1764      34105      0.707      0.498      0.537      0.308
                   car       1748      19633      0.815      0.759      0.804       0.51
                person        628       2450      0.731      0.593      0.653      0.333
          traffic sign       1461       6339      0.695      0.638      0.667      0.361
         traffic light        817       3787      0.709      0.662      0.685      0.295


18:55:34 | INFO     | yolov11: {'map50': 0.5372960444819949, 'map50_95': 0.3076155747350833, 'precision': 0.7072488246854561, 'recall': 0.49775074322272894}
18:55:34 | INFO     | Evaluating yolov12 on clear-day val...


Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv12m summary (fused): 169 layers, 20,112,622 parameters, 0 gradients, 67.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2437.3±683.0 MB/s, size: 74.2 KB)
val: Scanning /content/data_prepared/yolo/clear_day/val/labels.cache... 1764 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1764/1764 672.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 8.2it/s 13.5s0.2s
                   all       1764      34105      0.725      0.468      0.517      0.296
                   car       1748      19633      0.842       0.74      0.801      0.507
                person        628       2450      0.764      0.573      0.647      0.329
          traffic sign       1461       6339      0.734      0.607      0.665      0.357
         traffic light        817       3787      0.747      0.642      0.683      0.297


18:55:52 | INFO     | yolov12: {'map50': 0.5165867920996632, 'map50_95': 0.29587662954723926, 'precision': 0.7253634741154971, 'recall': 0.4677247166601256}
18:55:52 | INFO     | Evaluating rtdetr on clear-day val...


Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary: 310 layers, 32,004,290 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2281.1±823.8 MB/s, size: 73.3 KB)
val: Scanning /content/data_prepared/yolo/clear_day/val/labels.cache... 1764 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1764/1764 672.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 6.2it/s 17.9s0.1s
                   all       1764      34105      0.615      0.514      0.545      0.308
                   car       1748      19633      0.829      0.764      0.827       0.51
                person        628       2450      0.705      0.611      0.664      0.338
          traffic sign       1461       6339        0.7      0.676      0.698      0.377
         traffic light        817       3787      0.744      0.684      0.713        0.3
      

18:56:15 | INFO     | rtdetr: {'map50': 0.5451321039161057, 'map50_95': 0.30754108257566865, 'precision': 0.6154753054597959, 'recall': 0.5143807470035144}
18:56:15 | INFO     | Evaluating rfdetr on clear-day val...


loading annotations into memory...
Done (t=0.23s)
creating index...
index created!
Loading and preparing results...
DONE (t=3.37s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=93.07s).
Accumulating evaluation results...
DONE (t=10.31s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.323
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.563
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.314
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.136
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.377
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.701
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.244
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.438
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

19:00:44 | INFO     | rfdetr: {'map50': 0.5634430663672136, 'map50_95': 0.32277207985052825}


Saved scores and per-class AP to /content/drive/MyDrive/FON/master_rad/results/clear_day


In [4]:
try:
    import fvcore  # noqa: F401
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', 'fvcore', '-q'], check=True)

dummy = torch.zeros(1, 3, 640, 640).to(DEVICE)
hw_metrics = {}

for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
    vram  = measure_vram(model.model, dummy, device=DEVICE)
    flops = count_flops_and_params(model.model, dummy)
    speed = measure_inference_speed(model.model, dummy, device=DEVICE)
    hw_metrics[name] = {**vram, **flops, **speed}
    logger.info(f'{name} hw: {hw_metrics[name]}')

hw_metrics['rfdetr'] = measure_rfdetr_hardware(rfdetr, device=DEVICE, resolution=576)
logger.info(f"rfdetr hw: {hw_metrics['rfdetr']}")

(RESULTS_DIR / 'hw_metrics.json').write_text(json.dumps(hw_metrics, indent=2))
print('Saved hw_metrics to', RESULTS_DIR / 'hw_metrics.json')

19:00:46 | INFO     | yolov11 hw: {'peak_vram_mb': 666.07, 'peak_vram_gb': 0.6505, 'gflops': 33.906, 'params_m': 0.0, 'ms_per_frame': 10.948, 'fps': 91.34}
19:00:48 | INFO     | yolov12 hw: {'peak_vram_mb': 666.11, 'peak_vram_gb': 0.6505, 'gflops': 35.215, 'params_m': 0.0, 'ms_per_frame': 14.877, 'fps': 67.22}
19:00:53 | INFO     | rtdetr hw: {'peak_vram_mb': 658.23, 'peak_vram_gb': 0.6428, 'gflops': 52.693, 'params_m': 0.0, 'ms_per_frame': 36.767, 'fps': 27.2}
19:00:56 | INFO     | rfdetr hw: {'peak_vram_mb': 607.84, 'peak_vram_gb': 0.5936, 'gflops': nan, 'params_m': nan, 'ms_per_frame': 32.106, 'fps': 31.15}


Saved hw_metrics to /content/drive/MyDrive/FON/master_rad/results/clear_day/hw_metrics.json
